## Alteracoes para acelerar o treino e a inferencia

- **Resolucao**: `img_height`/`img_width` reduzidos de 61 para 32 (a tarefa e classificar a cor de fundo predominante, nao reconhecer a letra -- o fundo ocupa ~83% dos pixels em uma amostra real).
- **`interpolation="area"`**: reamostragem mais adequada para reduzir imagens (evita aliasing); o mesmo metodo foi usado em `compvision.py` para manter treino e inferencia consistentes.
- **`Flatten()` -> `GlobalAveragePooling2D()`**: elimina a camada `Dense` gigante que vinha do flatten (a maior parte dos parametros do modelo) sem perder informacao relevante para classificar cor.
- **`EarlyStopping`**: interrompe o treino quando a acuracia de validacao para de melhorar, em vez de sempre rodar todas as epocas.

**Depois de rodar este notebook, `modelo.keras` tera uma arquitetura nova -- e preciso treinar de novo antes de usar com `compvision.py`** (que tambem foi atualizado para esperar entradas 32x32).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
import tensorflow as tf
import pathlib

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

In [ ]:
data_dir = pathlib.Path("../data/images/")

In [ ]:
image_count = len(list(data_dir.glob('*/*.png')))
print(image_count)

In [ ]:
yellow = list(data_dir.glob('amarelo/*'))
PIL.Image.open(str(yellow[0]))

In [ ]:
batch_size = 64
# Resolucao reduzida (61 -> 32): a tarefa e classificar a COR de fundo
# do quadrado (amarelo/preto/verde), nao reconhecer a letra. Na imagem
# de amostra enviada, o fundo ja ocupa ~83% dos pixels, entao uma
# resolucao bem menor preserva o sinal relevante e acelera tanto o
# treino (menos pixels por convolucao) quanto a inferencia. Se a
# acuracia cair ao testar, aumente de volta (ex.: 48) e retreine.
img_height = 32
img_width = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  interpolation="area",
  batch_size=batch_size)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  interpolation="area",
  batch_size=batch_size)

In [ ]:
class_names = train_ds.class_names
print(class_names)

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
for image_batch, labels_batch in train_ds:
  print(image_batch.shape)
  print(labels_batch.shape)
  break

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
normalization_layer = layers.Rescaling(1./255)

In [ ]:
normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]
# Notice the pixel values are now in `[0,1]`.
print(np.min(first_image), np.max(first_image))

In [ ]:
num_classes = len(class_names)

# Flatten() -> GlobalAveragePooling2D(): o Flatten() gerava um vetor
# gigante (H x W x canais) alimentando a camada Dense seguinte, que
# concentrava a maior parte dos parametros (e do tempo de computacao)
# do modelo inteiro. GlobalAveragePooling2D reduz cada mapa de ativacao
# a um unico valor (a media) -- o que faz sentido aqui, ja que estamos
# classificando uma cor de fundo predominante, nao uma forma -- e
# derruba drasticamente o numero de parametros da camada Dense sem
# perder a informacao relevante para essa tarefa.
model = Sequential([
  layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
  layers.Conv2D(8, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.GlobalAveragePooling2D(),
  layers.Dense(16, activation='relu'),
  layers.Dense(num_classes)
])

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
import time

epochs = 30
# EarlyStopping interrompe o treino assim que a acuracia de validacao
# para de melhorar por algumas epocas seguidas (em vez de sempre rodar
# as 30 epocas inteiras) e devolve os melhores pesos vistos -- em geral
# treina mais rapido, sem piorar o resultado final.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True,
)

inicio = time.time()
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs,
  callbacks=[early_stopping]
)
print(f"Tempo total de treino: {time.time() - inicio:.1f}s ({len(history.epoch)} epocas)")

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
caminho_imagem = pathlib.Path("../data/images/amarelo/a.png")
img = tf.keras.utils.load_img(caminho_imagem, target_size=(img_height, img_width))
img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

previsoes = model.predict(img_array)
print(previsoes)
model.save("modelo.keras")